### Callable对象 ： 可以被当作函数调用的对象
- 函数是典型的callable对象
- 定义了__call__()方法的类的实例
- 匿名函数 lambda

In [ ]:
def sum(a,b):
    return a+b

callable(sum)

### 函数的作用域
- 全局作用域：
- 局部作用域：
- 外层作用域：


In [ ]:
GLOBAL = 100  # 全局作用域

def divide_num(num):
    GLOBAL = 50
    return GLOBAL/num
print(divide_num(2))
print(GLOBAL)

In [ ]:
from typing import TypedDict

class TextChunkScheme(TypedDict):
    tokens:int
    content:str
    full_doc_id:str
    chunk_order_index:int

### async with 异步上下文管理器
async with CLASS 该 CLASS必须实现__aenter__()和__aexit__()方法


In [ ]:
import asyncio

class AsyncDatabase:
    """模拟异步数据库连接"""

    async def __aenter__(self):
        print("🔗 异步建立数据库连接...")
        await asyncio.sleep(1)  # 模拟连接耗时
        print("✅ 连接成功！")
        return self  # 返回资源给as后面的变量

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("🔒 异步关闭数据库连接...")
        await asyncio.sleep(0.5)  # 模拟关闭耗时
        print("✅ 连接已关闭！")

    async def query(self, sql):
        print(f"📊 执行查询: {sql}")
        await asyncio.sleep(0.5)
        return ["数据1", "数据2"]

# 使用 async with
async def main():
    async with AsyncDatabase() as db:  # 自动管理连接生命周期
        result = await db.query("SELECT * FROM users")
        print(f"查询结果: {result}")
    # 出了with块，自动调用__aexit__关闭连接

await main()

### asyncio信号量机制 semaphore


In [ ]:
import asyncio,time

class AsyncFileDownloader:

    def __init__(self,max_limitation=2):
        self.semaphore = asyncio.Semaphore(max_limitation)
        self.max_limitation = max_limitation

    async def __aenter__(self):
        print(f"🚀 下载器启动，并发限制: {self.max_limitation}")
        await asyncio.sleep(1)
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("下载器关闭中")
        await asyncio.sleep(1)
        print("🏁 下载器关闭")

    async def download(self,file_id):
        async with self.semaphore:
            print(f"⬇️开始下载文件{file_id}...")
            await asyncio.sleep(1)
            print(f"✅文件{file_id}完成")
            return f"{file_id}的内容"
async def main():
    file_ids = [1, 2, 3, 4, 5]  # 5个文件要下载

    start = time.time()

    # async with 确保下载器正确初始化和清理
    async with AsyncFileDownloader(2) as downloader:
        # 创建5个下载任务，同时启动
        tasks = [downloader.download(fid) for fid in file_ids]
        # gather等待所有任务完成
        results = await asyncio.gather(*tasks)

    print(f"\n📦 所有结果: {results}")
    print(f"⏱️ 总耗时: {time.time() - start:.1f}秒")
    print(f"💡 理论最短耗时: {len(file_ids) / 2 * 1:.1f}秒 (5个任务，每次2个并发，每个1秒)")

# 运行
await main()

### 生成器（generator） VS 迭代器（iterator）

In [11]:
# =============== 理解什么时可迭代对象，什么是迭代器 =================
# 可迭代对象：python内置的list，str，dict都是可迭代对象
# 最基本的判断方法：我们可以使用for循环进行遍历 或者 判断有没有实现__iter__的魔法方法
my_list = [1,2,3]
my_str = "123"
my_dict = {"num_1":1,"num_2":2,"num_3":3}

print(f"{hasattr(my_list,'__iter__')}, {hasattr(my_dict,'__iter__')},{hasattr(my_str,'__iter__')}")

my_list_iterator = iter(my_list)
print(my_list_iterator.__next__(),type(my_list_iterator))
# =============== 什么是生成器 ==============
def get_number_gen(num):
    for i in range(num):
        yield i

def get_number_list(num):
    _list = []
    for i in range(num):
        _list.append(i)
    return _list

test_list_gen = get_number_gen(5)
test_list = get_number_list(5)

print(f"传统函数生成列表的结果{test_list},类型：{type(test_list)}")
print(f"生成器函数返回的生成器对象{test_list_gen},类型：{type(test_list_gen)}")
test_list_g = [num for num in test_list_gen]
# ============== 为什么使用生成器 ===============
import sys
# 在处理大数据的时候节省内存
list_size = sys.getsizeof(test_list)
list_size_gen = sys.getsizeof(test_list_gen)
print(f"列表所占内存：{list_size},生成器所占内存：{list_size_gen}")
print(f"列表所占内存是生成器的{list_size/list_size_gen}倍")


True, True,True
1 <class 'list_iterator'>
传统函数生成列表的结果[0, 1, 2, 3, 4],类型：<class 'list'>
生成器函数返回的生成器对象<generator object get_number_gen at 0x00000209ABC875E0>,类型：<class 'generator'>
列表所占内存：120,生成器所占内存：200
列表所占内存是生成器的0.6倍


In [19]:
# =============== 生成器的高级特性 ===============
# 1.委托子生成器
def subgen():
    yield 1
    yield 2

def main_gen():
    yield "开始"
    yield from subgen()
    yield "结束"

print(list(main_gen()))

def counter():
    count = 0
    while True:
        increment = yield count
        if increment is None:
            increment = 1
        count += increment

count = counter()
count.__next__()
count.send(3)
count.__next__()
count.close()
count

['开始', 1, 2, '结束']


<generator object counter at 0x00000209ABA83400>

### python 网络世界的http库
#### http全名为hypertext transfer protocol 超文本传输协议
#### 请求 - 响应 两部分组成
---
请求包括：
- 请求行：方法 + 资源路径 + 协议版本 例如：GET user/name HTTP/1.1
- 请求头：包括
Content-Type: application/json
Cookie: sessionid=xxxx
Authorization: Bearer token
- 请求体：POST方法有的 例如传递的用户info信息
---
响应 对应 请求包括：
- 响应行： 版本协议 + 状态码 HTTP/1.1 200 ok
- 响应行：
- 响应体：

In [21]:
import http.client
import json

conn = http.client.HTTPSConnection("httpbin.org")

conn.request("GET", "/get?name=python&age=30")

response = conn.getresponse()

print(f"状态码: {response.status} {response.reason}")
print(f"响应头:")
for header, value in response.getheaders():
    print(f"  {header}: {value}")

body = response.read().decode('utf-8')
print(f"\n响应体:\n{body}")

状态码: 200 OK
响应头:
  Date: Tue, 24 Feb 2026 11:19:31 GMT
  Content-Type: application/json
  Content-Length: 295
  Connection: keep-alive
  Server: gunicorn/19.9.0
  Access-Control-Allow-Origin: *
  Access-Control-Allow-Credentials: true

响应体:
{
  "args": {
    "age": "30", 
    "name": "python"
  }, 
  "headers": {
    "Accept-Encoding": "identity", 
    "Host": "httpbin.org", 
    "X-Amzn-Trace-Id": "Root=1-699d8943-210e00237a19da065707efe1"
  }, 
  "origin": "36.143.26.49", 
  "url": "https://httpbin.org/get?name=python&age=30"
}



In [22]:
conn = http.client.HTTPSConnection("httpbin.org")

payload = json.dumps({
    "username": "admin",
    "password": "123456",
})

headers = {"Content-Type": "application/json","User-Agent": "MyPythonApp/1.0"}

conn.request("POST", "/post", payload, headers)

response = conn.getresponse()

body = json.loads(response.read().decode('utf-8'))

print("POST请求的结果：")
print(f"数据字段{body["data"]}")
print(f"解析字段{body["json"]}")

POST请求的结果：
数据字段{"username": "admin", "password": "123456"}
解析字段{'password': '123456', 'username': 'admin'}


#### 状态码列表
| 方法         | 用途       | 幂等性 | 示例      |
| ---------- | -------- | --- | ------- |
| **GET**    | 获取资源     | ✅   | 获取用户信息  |
| **POST**   | 创建资源     | ❌   | 提交表单、登录 |
| **PUT**    | 更新资源（完整） | ✅   | 修改用户资料  |
| **PATCH**  | 更新资源（部分） | ❌   | 修改用户昵称  |
| **DELETE** | 删除资源     | ✅   | 删除账号    |

In [25]:
status_codes = {
    200: "OK - 请求成功",
    201: "Created - 资源创建成功",
    400: "Bad Request - 请求参数错误",
    401: "Unauthorized - 未授权",
    403: "Forbidden - 禁止访问",
    404: "Not Found - 资源不存在",
    500: "Internal Server Error - 服务器内部错误",
    502: "Bad Gateway - 网关错误",
    503: "Service Unavailable - 服务不可用"
}

def check_status(url):
    """检查URL状态码"""
    try:
        conn = http.client.HTTPSConnection("httpbin.org")
        conn.request("GET", url)
        response = conn.getresponse()

        status = response.status
        description = status_codes.get(status, "未知状态码")

        print(f"URL: {url}")
        print(f"状态码: {status} - {description}")

        # 根据状态码处理
        if 200 <= status < 300:
            print("请求成功")
            return True
        elif 400 <= status < 500:
            print("客户端错误")
            return False
        elif 500 <= status < 600:
            print("服务器错误")
            return False

    except Exception as e:
        print(f"请求异常: {e}")
        return False
    finally:
        conn.close()

check_status("/status/403")

URL: /status/403
状态码: 403 - Forbidden - 禁止访问
客户端错误


False